# Lakebase Support Ticketing — Setup Notebook

> **Note:** This notebook is provided as a reference for setting up the Lakebase schema and seed data via Python/psycopg2.  
> In this project the DDL and DML were executed directly in the **Lakebase SQL Editor** inside the Databricks workspace.  
> Use this notebook if you need to re-run setup programmatically or in a CI/CD context.

**Lakebase project:** `support-ticketing`  
**Branch:** `production`  
**Database:** `databricks_postgres`  
**Auth:** OAuth via Databricks PAT  

## Cell 1 — Install psycopg2
Lakebase speaks the standard Postgres wire protocol, so psycopg2 is all we need.

In [ ]:
%pip install psycopg2-binary
dbutils.library.restartPython()

## Cell 2 — Set the connection string

**Where to find it:**
- Lakebase project → Overview → Roles & Databases
- Auth type: OAuth — use a Databricks PAT as the password

**Format:**
```
postgresql://token:<YOUR-PAT>@dbc-291b687e-da89.cloud.databricks.com:5432/databricks_postgres
```

> ⚠️ Never commit the PAT to GitHub. Use `dbutils.secrets.get()` in production.

In [ ]:
import os

# Option A: paste directly (dev only — never commit)
LAKEBASE_CONN = "postgresql://token:<YOUR-PAT>@dbc-291b687e-da89.cloud.databricks.com:5432/databricks_postgres"

# Option B: read from Databricks secrets (recommended for production)
# LAKEBASE_CONN = dbutils.secrets.get(scope="lakebase", key="conn_string")

## Cell 3 — Test the connection

In [ ]:
import psycopg2

def get_conn():
    return psycopg2.connect(LAKEBASE_CONN)

try:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT version();")
    version = cur.fetchone()
    print(f"✅ Connected to Lakebase!")
    print(f"   Postgres version: {version[0]}")
    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    raise

## Cell 4 — Create schema
Creates `tickets` and `ticket_messages` tables with indexes.  
`DROP TABLE IF EXISTS` makes this safe to re-run during development.

In [ ]:
DDL = """
DROP TABLE IF EXISTS ticket_messages;
DROP TABLE IF EXISTS tickets;

CREATE TABLE tickets (
    ticket_id   SERIAL       PRIMARY KEY,
    title       TEXT         NOT NULL,
    status      TEXT         NOT NULL DEFAULT 'open',
    priority    TEXT         NOT NULL DEFAULT 'medium',
    category    TEXT,
    created_by  TEXT         NOT NULL,
    created_at  TIMESTAMPTZ  NOT NULL DEFAULT NOW()
);

CREATE TABLE ticket_messages (
    message_id   SERIAL       PRIMARY KEY,
    ticket_id    INT          NOT NULL REFERENCES tickets(ticket_id) ON DELETE CASCADE,
    message_text TEXT         NOT NULL,
    author       TEXT         NOT NULL,
    created_at   TIMESTAMPTZ  NOT NULL DEFAULT NOW()
);

CREATE INDEX idx_messages_ticket_id ON ticket_messages(ticket_id);
CREATE INDEX idx_tickets_status     ON tickets(status);
CREATE INDEX idx_tickets_priority   ON tickets(priority);
"""

try:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(DDL)
    conn.commit()
    print("✅ Schema created: tickets, ticket_messages, 3 indexes")
    cur.close()
    conn.close()
except Exception as e:
    conn.rollback()
    print(f"❌ Schema creation failed: {e}")
    raise

## Cell 5 — Insert seed data
4 tickets across 3 statuses (`open`, `in_progress`, `resolved`) with 11 messages total.

In [ ]:
SEED_TICKETS = """
INSERT INTO tickets (title, status, priority, category, created_by, created_at) VALUES
('Databricks cluster auto-terminates during pipeline run',        'open',        'high',   'infra',  'jay.dolai',    NOW() - INTERVAL '3 days'),
('Unable to access Unity Catalog schema after permission update', 'in_progress', 'high',   'access', 'priya.sharma', NOW() - INTERVAL '2 days'),
('Delta table OPTIMIZE job taking longer than expected',          'resolved',    'medium', 'data',   'ravi.kumar',   NOW() - INTERVAL '5 days'),
('Lakebase connection string not recognized in Databricks App',   'open',        'medium', 'infra',  'jay.dolai',    NOW() - INTERVAL '1 day');
"""

SEED_MESSAGES = """
INSERT INTO ticket_messages (ticket_id, message_text, author, created_at) VALUES
(1, 'The cluster keeps shutting down at the 2-hour mark even though auto-termination is set to 240 minutes.', 'jay.dolai',    NOW() - INTERVAL '3 days'),
(1, 'Checked the cluster event log — looks like spot instance preemption. Try switching to on-demand workers.', 'support.bot', NOW() - INTERVAL '2 days 18 hours'),
(1, 'Switched to on-demand workers. Will monitor tonight''s run and update tomorrow.', 'jay.dolai',             NOW() - INTERVAL '2 days'),
(2, 'After the admin ran an access policy update, I can no longer query gold.sales_summary. Getting PERMISSION_DENIED.', 'priya.sharma', NOW() - INTERVAL '2 days'),
(2, 'Confirmed. The policy update revoked data_reader group privilege on the gold schema. Fix in progress.',    'admin.team',  NOW() - INTERVAL '1 day 21 hours'),
(2, 'Partial fix applied. Please test read access and confirm. Still auditing other affected schemas.',         'admin.team',  NOW() - INTERVAL '1 day'),
(3, 'Our weekly OPTIMIZE + ZORDER on the events table (800 GB) is taking 6+ hours. Used to finish in 90 min.', 'ravi.kumar',  NOW() - INTERVAL '5 days'),
(3, 'The new cluster policy caps workers at 4 nodes. Previous runs used 12. Recommend partition-range OPTIMIZE.', 'support.bot', NOW() - INTERVAL '4 days'),
(3, 'Switched to partition-range OPTIMIZE by month. Job now completes in 95 minutes. Marking resolved!',        'ravi.kumar',  NOW() - INTERVAL '3 days'),
(4, 'When I set LAKEBASE_CONN as env var and call psycopg2.connect(), I get: connection refused on port 5432.', 'jay.dolai',   NOW() - INTERVAL '1 day'),
(4, 'Use the internal Lakebase hostname and confirm the App is in the same workspace region.',                  'support.bot', NOW() - INTERVAL '20 hours');
"""

try:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(SEED_TICKETS)
    cur.execute(SEED_MESSAGES)
    conn.commit()
    print("✅ Seed data inserted")
    cur.close()
    conn.close()
except Exception as e:
    conn.rollback()
    print(f"❌ Seed failed: {e}")
    raise

## Cell 6 — Verify
Confirms all tickets exist with the correct message counts and statuses.

In [ ]:
import pandas as pd

VERIFY_QUERY = """
SELECT
    t.ticket_id,
    t.title,
    t.status,
    t.priority,
    t.category,
    t.created_by,
    COUNT(m.message_id) AS message_count
FROM tickets t
LEFT JOIN ticket_messages m ON t.ticket_id = m.ticket_id
GROUP BY t.ticket_id, t.title, t.status, t.priority, t.category, t.created_by
ORDER BY t.ticket_id;
"""

conn = get_conn()
df = pd.read_sql(VERIFY_QUERY, conn)
conn.close()

print(f"✅ Verification complete")
print(f"   Tickets  : {len(df)}")
print(f"   Statuses : {df['status'].unique().tolist()}")
print(f"   Messages : {int(df['message_count'].sum())} total")
print()
display(df)

## ✅ Setup complete

Expected output from Cell 6:

| ticket_id | title | status | priority | message_count |
|---|---|---|---|---|
| 1 | Databricks cluster auto-terminates... | open | high | 3 |
| 2 | Unable to access Unity Catalog... | in_progress | high | 3 |
| 3 | Delta table OPTIMIZE job... | resolved | medium | 3 |
| 4 | Lakebase connection string... | open | medium | 2 |

Next step → deploy `app/app.py` as a Databricks App (Phase 3 & 4).